In [ ]:
## Pothan Tang, 8/15/25
## Compute DC potential taking into account M2 grounding

import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt

## region of interest
res = 1*c.um; # resolution of potential 
x_max = 100*c.um;
y_max = 100*c.um;
z_max = 200*c.um;
x = np.linspace(-1*x_max,x_max, int(2*x_max/res+1), endpoint=True); 
y = np.linspace(-1*y_max,y_max, int(2*y_max/res+1), endpoint=True);  
z = np.linspace(0,z_max, int(z_max/res+1), endpoint=True);  
phi = np.zeros((len(x),len(y),len(z)), dtype=np.float32); # potential initialized to zero

## Functions
# (x,y,z) = coordinate of sample
# (xik,yik,z0) = ith corner coordinates of kth electrode
# vk = voltage applied to kth electrode
def potential_term(x,y,z,xik,yik,z0):
    num = (xik-x)*(yik-y); # numerator
    den = (z-z0)*math.sqrt((z-z0)**2+(xik-x)**2+(yik-y)**2); # denominator
    return num/den

def dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,vk):
    term1 = math.atan(potential_term(x,y,z,x2k,y2k,z0))
    term2 = math.atan(potential_term(x,y,z,x1k,y2k,z0))
    term3 = math.atan(potential_term(x,y,z,x2k,y1k,z0))
    term4 = math.atan(potential_term(x,y,z,x1k,y1k,z0))
    return (vk/(2*math.pi))*(term1-term2-term3+term4)

def dc_potential_total(x,y,z,xy1k,xy2k,z0,vk):
    # extract individual (xik,yik),vk values from array
    # all arrays must be same size
    # sum potentials from all electrodes
    pot = 0
    for i in range (0,len(vk)):
        x1k = xy1k[i][0]
        y1k = xy1k[i][1]
        x2k = xy2k[i][0]
        y2k = xy2k[i][1]
        v = vk[i]
        if(i==19 or i==39): # make sure Q39-40 are on the M4 layer
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,0,v); # z0=0 for Q39-40
        else:
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,v); # z0 = -11.8um for Q1-38
        pot+=new_pot
    return pot

## Compute Potential
for p in range(0,len(x)):
    xc = x[p]
    for q in range (0,len(y)):
        yc = y[q]
        for r in range (0,len(z)):
            zc = z[r]
            phi[p][q][r] = dc_potential_total(xc,yc,zc,c.xy1k,c.xy2k,c.z0,c.vk)

phi_reshaped = phi.reshape(int(2*x_max/res+1),-1)
np.savetxt("dc_potential_zoomed", phi_reshaped)

/tmp/ipykernel_14448/1674142056.py:26: RuntimeWarning: divide by zero encountered in scalar divide
  return num/den
